# Model Context Protocol for Enterprise System Integration

## What you will build

You will connect an assistant to a company's reporting database, the one the finance team uses to
chase unpaid invoices. The assistant never touches the database itself. A small server sits in
front of it and offers a fixed set of safe queries, and your client carries each request from the
model to that server and brings the answer back.

The model context protocol is the agreement those two programs speak, and every part of it is easy
to get slightly wrong. The diagram shows the three mistakes this course stops: a failed query that
the client counts as a success, a log line that breaks the stream of replies, and a query that
writes to the database when it should only read.

![What you will build](images/mcp-overview.svg)

## Step 0: Set up the client and the model

Every model call in this notebook goes through the repository's own client. Without an API key it
answers from responses recorded in real runs, so you can follow the course for free, and with a key
it calls the model live.

In [1]:
import json
import os
import pathlib
import queue
import socket
import sqlite3
import subprocess
import sys
import threading
import time
import warnings

import requests
from vault import get_client, load_env, model_for

load_env()
client = get_client("11-model-context-protocol/01-connect-an-mcp-client-to-a-database")
MODEL = model_for("default")

print(f"Client ready. Every model request in this notebook uses {MODEL}.")

Client ready. Every model request in this notebook uses google/gemini-2.5-flash-lite.


## Step 1: Create the reporting database the server will read

The server needs a real database behind it, so we start with a small reporting database for the
finance team. In production this would be the company's PostgreSQL server. Here it is a SQLite
file, used as a local stand-in because it needs no install, and nothing the course teaches about
the server or the client changes with the database behind it.

In [2]:
DATABASE_PATH = pathlib.Path("reporting.db").resolve()
DATABASE_PATH.unlink(missing_ok=True)

setup = sqlite3.connect(DATABASE_PATH)
setup.executescript("""
CREATE TABLE customers(customer_id TEXT PRIMARY KEY, name TEXT, region TEXT);
CREATE TABLE invoices(invoice_id TEXT PRIMARY KEY, customer_id TEXT,
                      amount_cents INTEGER, status TEXT, due_date TEXT);
CREATE TABLE payroll(employee TEXT, salary_cents INTEGER);
INSERT INTO customers VALUES ('CUST-101', 'Nordlicht GmbH', 'EMEA'),
    ('CUST-102', 'Harbour Foods', 'EMEA'), ('CUST-201', 'Pine Street Labs', 'AMER');
INSERT INTO invoices VALUES ('INV-5001', 'CUST-101', 1250000, 'overdue', '2026-07-31'),
    ('INV-5002', 'CUST-102', 480000, 'overdue', '2026-08-15'),
    ('INV-5003', 'CUST-101', 90000, 'paid', '2026-06-30'),
    ('INV-5004', 'CUST-201', 730000, 'overdue', '2026-08-01');
INSERT INTO payroll VALUES ('j.smith', 9800000);
""")
setup.commit()
setup.close()
os.environ["REPORTING_DB"] = str(DATABASE_PATH)   # the server reads its path from here

The payroll table shares the database by accident, which is common in real companies. Nothing in
this course should ever read it, and the next cell counts the rows so we can check that later.

In [3]:
def count_rows(table):
    """Rows in one table, read on a connection of its own."""
    with sqlite3.connect(DATABASE_PATH) as database:
        return database.execute(f"SELECT count(*) FROM {table}").fetchone()[0]


for table in ("customers", "invoices", "payroll"):
    print(f"{table:10} {count_rows(table)} rows")

customers  3 rows
invoices   4 rows
payroll    1 rows


## Step 2: Write an MCP server with tools, resources and a prompt

We now write the server that stands between the model and the database. An **MCP server** offers
three kinds of thing, called the **MCP primitives**: tools that the model can call, resources that
the client reads by address, and prompts that a person picks from a menu. The first part of the
file opens the database and defines the one function that runs every query.

![Write an MCP server with tools, resources and a prompt](images/server-transport-step-1.svg)

In [4]:
%%writefile reporting_server.py
import contextlib
import json
import os
import sqlite3
import sys

from mcp.server.fastmcp import FastMCP

DATABASE_PATH = os.environ["REPORTING_DB"]
LOG_STREAM = sys.stdout if os.environ.get("REPORTING_LOG") == "stdout" else sys.stderr
server = FastMCP("reporting", log_level="WARNING", stateless_http=True, json_response=True,
                 port=int(os.environ.get("REPORTING_PORT", "8000")))


def log_query(message):
    print(f"[reporting] {message}", file=LOG_STREAM, flush=True)


def fetch_rows(sql, parameters):
    """Run one read on a connection that the SQLite driver refuses to write through."""
    log_query(f"{sql.split()[0]} with {parameters}")
    uri = f"file:{DATABASE_PATH}?mode=ro"
    with contextlib.closing(sqlite3.connect(uri, uri=True)) as database:
        cursor = database.execute(sql, parameters)
        names = [column[0] for column in cursor.description]
        return [dict(zip(names, row)) for row in cursor.fetchall()]

Writing reporting_server.py


The `mode=ro` flag opens the file read only, so the driver rejects any write before it reaches the
data. On PostgreSQL the same guarantee comes from a database role that has been granted SELECT and
nothing else. `log_query` writes to stderr unless `REPORTING_LOG` says otherwise, and Step 7 shows
why that choice matters.

The tools are the queries the model may ask for. Each one is a fixed SQL statement with its values
bound as parameters, so the model chooses the values but never writes the SQL.

In [5]:
%%writefile -a reporting_server.py


@server.tool()
def list_overdue_invoices(region_code: str, min_amount_cents: int = 0) -> list[dict]:
    """Overdue invoices for one sales region (EMEA, AMER or APAC), largest first."""
    return fetch_rows(
        "SELECT invoice_id, name, amount_cents, due_date FROM invoices "
        "JOIN customers USING (customer_id) WHERE region = ? AND status = 'overdue' "
        "AND amount_cents >= ? ORDER BY amount_cents DESC LIMIT 20",
        (region_code, min_amount_cents))


@server.tool()
def list_customer_invoices(customer_id: str) -> list[dict]:
    """Every invoice for one customer id, such as CUST-101, newest first."""
    return fetch_rows(
        "SELECT invoice_id, amount_cents, status, due_date FROM invoices "
        "WHERE customer_id = ? ORDER BY due_date DESC LIMIT 50", (customer_id,))

Appending to reporting_server.py


The resources and the prompt finish the file. A resource is read by its address, such as
`reporting://customers/CUST-101`, and the protocol has no method that writes one. The last two lines
let the command line choose how the server talks to its client, which Step 8 uses.

In [6]:
%%writefile -a reporting_server.py


@server.resource("reporting://schema")
def describe_schema() -> str:
    """The tables and columns this server reads. Payroll is not one of them."""
    return json.dumps({"customers": ["customer_id", "name", "region"],
                       "invoices": ["invoice_id", "customer_id", "amount_cents",
                                    "status", "due_date"]})


@server.resource("reporting://customers/{customer_id}")
def read_customer(customer_id: str) -> str:
    """One customer record, addressed by its id."""
    rows = fetch_rows("SELECT customer_id, name, region FROM customers "
                      "WHERE customer_id = ? LIMIT 1", (customer_id,))
    return json.dumps(rows[0] if rows else {})


@server.prompt()
def summarize_overdue(region_code: str, min_amount_cents: str) -> str:
    """Wording for a finance summary of one region's larger overdue invoices."""
    return (f"Summarise the overdue invoices of at least {min_amount_cents} cents "
            f"for {region_code} in three sentences for the finance team.")


if __name__ == "__main__":
    server.run(sys.argv[1] if len(sys.argv) > 1 else "stdio")

Appending to reporting_server.py


Before any client exists, we can import the file as a module and ask its own query function to
delete every invoice. The library prints a harmless warning about its own settings on import, so
the cell hides that one warning.

In [7]:
warnings.filterwarnings("ignore", message="Field 'lifespan'")
import reporting_server

try:
    reporting_server.fetch_rows("DELETE FROM invoices", ())
except sqlite3.OperationalError as error:
    print(f"refused by the driver: {error}")
print(f"invoices still in the table: {count_rows('invoices')}")

[reporting] DELETE with ()


refused by the driver: attempt to write a readonly database
invoices still in the table: 4


## Step 3: Start the server over stdio and exchange JSON-RPC messages

A client and a server need a **transport**, which is how two programs carry messages to each
other. The simplest one is **stdio**, which means talking over a program's own input and output
streams: the client starts the server as a child process, writes requests to its stdin and reads
replies from its stdout.

![Start the server over stdio and exchange JSON-RPC messages](images/server-transport-step-2.svg)

Every message is **JSON-RPC 2.0**, a small convention for calling functions over a stream using
JSON. A request carries `jsonrpc`, an `id`, a `method` and its `params`, and the reply carries the
same `id` with either a `result` or an `error`. Over stdio each message is one line, so the client
treats any line that is not JSON as a broken stream.

In [8]:
def parse_json_rpc_line(line):
    """Every line on the server's stdout must be one JSON-RPC message, and nothing else."""
    try:
        return json.loads(line)
    except json.JSONDecodeError:
        raise ValueError(f"stdout carried a line that is not JSON-RPC: {line.strip()!r}") from None

`StdioConnection` starts the server and copies each line it prints into a queue, so the client can
wait for a reply with a time limit instead of blocking forever.

In [9]:
class StdioConnection:
    """The server run as a child process, spoken to over its stdin and stdout."""

    def __init__(self, log_to="stderr"):
        env = {**os.environ, "REPORTING_LOG": log_to, "PYTHONWARNINGS": "ignore"}
        self.process = subprocess.Popen(
            [sys.executable, "reporting_server.py"], text=True, env=env,
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        self.stdout_lines = queue.Queue()
        threading.Thread(target=self.copy_stdout, daemon=True).start()
        self.last_id = 0

    def copy_stdout(self):
        for line in self.process.stdout:
            self.stdout_lines.put(line)

    def exchange(self, message, timeout=10):
        """Write one line to stdin. A request then waits for one line back."""
        self.process.stdin.write(json.dumps(message) + "\n")
        self.process.stdin.flush()
        return parse_json_rpc_line(self.stdout_lines.get(timeout=timeout)) if "id" in message else None

    def close(self):
        """Close stdin, which tells the server to exit, and return what it wrote to stderr."""
        self.process.stdin.close()
        self.process.wait(timeout=10)
        return self.process.stderr.read()

`send_request` numbers each request and checks the reply against it. `start_session` performs the
handshake that every MCP connection opens with: an `initialize` request, then a notification, which
is a message with no `id` that gets no reply.

In [10]:
def send_request(connection, method, params=None):
    """Send one JSON-RPC request and return its result. A JSON-RPC error raises."""
    connection.last_id += 1
    request = {"jsonrpc": "2.0", "id": connection.last_id, "method": method}
    if params is not None:
        request["params"] = params
    reply = connection.exchange(request)
    assert reply["id"] == request["id"], f"reply {reply['id']} answers a different request"
    if "error" in reply:
        raise RuntimeError(f"JSON-RPC error {reply['error']['code']}: {reply['error']['message']}")
    return reply["result"]


def start_session(connection):
    """The handshake: initialize, then the notification that says the client is ready."""
    result = send_request(connection, "initialize", {
        "protocolVersion": "2025-06-18", "capabilities": {},
        "clientInfo": {"name": "finance-reporting-client", "version": "1.0"}})
    connection.exchange({"jsonrpc": "2.0", "method": "notifications/initialized"})
    return result

The next cell starts the server, completes the handshake, and then sends one request by hand, so
you can see the exact text that crosses the pipe in each direction.

In [11]:
connection = StdioConnection()
server_info = start_session(connection)
print(f"handshake done with {server_info['serverInfo']['name']}, "
      f"protocol {server_info['protocolVersion']}\n")

request = {"jsonrpc": "2.0", "id": 99, "method": "tools/list"}
reply = connection.exchange(request)
print(f"sent     : {json.dumps(request)}")
print(f"received : {json.dumps(reply)[:140]}...")

handshake done with reporting, protocol 2025-06-18

sent     : {"jsonrpc": "2.0", "id": 99, "method": "tools/list"}
received : {"jsonrpc": "2.0", "id": 99, "result": {"tools": [{"name": "list_overdue_invoices", "description": "Overdue invoices for one sales region (E...


The reply lists both tools. Look at the keys each tool carries, because the next steps depend on
them.

In [12]:
MCP_TOOLS = send_request(connection, "tools/list")["tools"]
for tool in MCP_TOOLS:
    print(f"{tool['name']:24} keys: {sorted(tool)}")

list_overdue_invoices    keys: ['description', 'inputSchema', 'name', 'outputSchema']
list_customer_invoices   keys: ['description', 'inputSchema', 'name', 'outputSchema']


## Step 4: Read resources and prompts, which the model never picks

Tools are one of three primitives, and the model picks only tools. The client application reads
resources by address, and a person picks prompts, so we fetch one of each here and put them where a
real client would.

In [13]:
resources = send_request(connection, "resources/list")["resources"]
templates = send_request(connection, "resources/templates/list")["resourceTemplates"]
prompts = send_request(connection, "prompts/list")["prompts"]

print("resources          :", [resource["uri"] for resource in resources])
print("resource templates :", [template["uriTemplate"] for template in templates])
print("prompts            :", [prompt["name"] for prompt in prompts])

resources          : ['reporting://schema']
resource templates : ['reporting://customers/{customer_id}']
prompts            : ['summarize_overdue']


The schema resource holds the database's **schema**, the written shape of the allowed data, with
its tables and columns. The client reads it once and places it in the system prompt, so the model
knows which tables exist without being able to query them freely.

In [14]:
def read_resource(connection, uri):
    """Fetch one resource by its address and return its text."""
    return send_request(connection, "resources/read", {"uri": uri})["contents"][0]["text"]


SCHEMA_TEXT = read_resource(connection, "reporting://schema")
SYSTEM_PROMPT = ("You answer finance questions from the company's reporting database. "
                 f"Always use your tools to fetch data. The tables are: {SCHEMA_TEXT}")

print(f"schema resource  : {SCHEMA_TEXT}")
print(f"customer record  : {read_resource(connection, 'reporting://customers/CUST-101')}")

schema resource  : {"customers": ["customer_id", "name", "region"], "invoices": ["invoice_id", "customer_id", "amount_cents", "status", "due_date"]}
customer record  : {"customer_id": "CUST-101", "name": "Nordlicht GmbH", "region": "EMEA"}


The analyst picks the summary prompt from a menu, then chooses a region and a minimum amount. The server fills in the
wording, and that text becomes the question the model answers in the next step.

In [15]:
prompt = send_request(connection, "prompts/get", {"name": "summarize_overdue", "arguments": {
    "region_code": "EMEA", "min_amount_cents": "500000"}})
ANALYST_REQUEST = prompt["messages"][0]["content"]["text"]

print(f"analyst request: {ANALYST_REQUEST}")

analyst request: Summarise the overdue invoices of at least 500000 cents for EMEA in three sentences for the finance team.


## Step 5: Hand the server's tools to the model the way a first client would

The model reads each tool as a name, a description and a schema for its arguments, which gives
their types and says which ones are required. MCP sends that schema in a field called `inputSchema`, and a call result says it failed in a field called `isError`. Both
are camel case, while Python code usually spells the same ideas `input_schema` and `is_error`.

![Hand the server's tools to the model the way a first client would](images/tool-calls-step-1.svg)

In [16]:
def translate_tool_naively(tool):
    """MCP tool to model tool, using the snake case names Python code expects."""
    return {"type": "function", "function": {
        "name": tool["name"], "description": tool.get("description", ""),
        "parameters": tool.get("input_schema", {"type": "object", "properties": {}})}}


def read_tool_result_naively(result):
    """Return the text for the model, and whether the call failed."""
    text = "\n".join(part["text"] for part in result["content"] if part["type"] == "text")
    return text, bool(result.get("is_error"))


naive_tools = [translate_tool_naively(tool) for tool in MCP_TOOLS]
print(f"parameters the model will see: {naive_tools[0]['function']['parameters']}")

parameters the model will see: {'type': 'object', 'properties': {}}


Both functions fall back to a default instead of failing, so nothing looks wrong yet. `run_agent`
is the loop from vault 1, except that each **tool call**, which is the model asking your code to
run a named function, is sent on to the MCP server as a `tools/call` request.

In [17]:
MAX_TURNS = 5
AUDIT_LOG = []      # the client's own record of each tool call
WIRE_RESULTS = []   # each tools/call result exactly as the server sent it


def run_agent(connection, model_tools, read_tool_result, question):
    """Call the model, send each tool call to the MCP server, and repeat until it answers."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question}]
    for turn in range(1, MAX_TURNS + 1):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=600, tools=model_tools, messages=messages)
        choice = response.choices[0]
        messages.append(choice.message.model_dump(exclude_none=True))
        if choice.finish_reason != "tool_calls":
            return choice.message.content
        for tool_call in choice.message.tool_calls:
            name, arguments = tool_call.function.name, json.loads(tool_call.function.arguments)
            result = send_request(connection, "tools/call", {"name": name, "arguments": arguments})
            text, failed = read_tool_result(result)
            AUDIT_LOG.append({"tool": name, "failed": failed})
            WIRE_RESULTS.append(result)
            rows = len(result.get("structuredContent", {}).get("result", []))
            print(f"  turn {turn}: {name}({arguments}) -> {rows} rows, failed={failed}")
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": text})
    raise RuntimeError(f"no answer after {MAX_TURNS} turns")

One answer proves nothing about a model, so the next cell sends the analyst's request three times.
After each run it compares the failures the client logged with the failures the server reported.

In [18]:
def report_run(attempt, answer):
    """Compare the client's audit log with what the server said on the wire."""
    logged = sum(entry["failed"] for entry in AUDIT_LOG)
    on_wire = sum(bool(result.get("isError")) for result in WIRE_RESULTS)
    print(f"attempt {attempt}: {len(AUDIT_LOG)} tool calls, client logged {logged} failures, "
          f"server sent isError on {on_wire}")
    print(f"answer: {answer[:160]}\n")


for attempt in range(1, 4):
    AUDIT_LOG.clear()
    WIRE_RESULTS.clear()
    report_run(attempt, run_agent(connection, naive_tools, read_tool_result_naively,
                                  ANALYST_REQUEST))

  turn 1: list_overdue_invoices({'region': 'EMEA'}) -> 0 rows, failed=False


attempt 1: 1 tool calls, client logged 0 failures, server sent isError on 1
answer: 
The total amount of overdue invoices in the EMEA region is 735000 cents. There is one overdue invoice that is over 500000 cents, with an amount of 510000 cents



  turn 1: list_overdue_invoices({'region': 'EMEA'}) -> 0 rows, failed=False


attempt 2: 1 tool calls, client logged 0 failures, server sent isError on 1
answer: I can list the overdue invoices for EMEA, but I cannot filter them by amount. I can also only provide the raw data, not a three-sentence summary.



  turn 1: list_overdue_invoices({'region': 'EMEA'}) -> 0 rows, failed=False


attempt 3: 1 tool calls, client logged 0 failures, server sent isError on 1
answer: The total amount of overdue invoices in the EMEA region is 746000 cents. There are two overdue invoices. The largest overdue invoice is for 500000 cents and the



Every attempt sent `region` instead of `region_code`, because the model could not see the
argument names, and the server answered each call with `isError` set to true. The client logged
all three calls as successes, and the model answered anyway. Attempts 1 and 3 invented totals of
735000 and 746000 cents, and attempt 2 said it could not filter by amount. The only overdue EMEA
invoice of at least 500000 cents is INV-5001, for 1250000 cents, so none of the three answers is
right.

## Step 6: Read inputSchema and isError by their wire names

The fix is to read the protocol's own field names in the one place that translates them, and to
fail loudly when a field that must be there is missing. A tool with no `inputSchema` raises a
`KeyError`, because a tool whose arguments the model cannot see should never be offered.

![Read inputSchema and isError by their wire names](images/tool-calls-step-2.svg)

In [19]:
def translate_tool(tool):
    """MCP tool to model tool. A tool with no inputSchema raises instead of being offered."""
    return {"type": "function", "function": {
        "name": tool["name"], "description": tool.get("description", ""),
        "parameters": tool["inputSchema"]}}


def read_tool_result(result):
    """Return the text for the model, and whether the call failed, read from isError."""
    text = "\n".join(part["text"] for part in result["content"] if part["type"] == "text")
    failed = bool(result.get("isError", False))   # the protocol says absent means false
    return (json.dumps({"error": text}) if failed else text), failed


model_tools = [translate_tool(tool) for tool in MCP_TOOLS]
print(f"parameters the model will see: {json.dumps(model_tools[0]['function']['parameters'])}")

parameters the model will see: {"properties": {"region_code": {"title": "Region Code", "type": "string"}, "min_amount_cents": {"default": 0, "title": "Min Amount Cents", "type": "integer"}}, "required": ["region_code"], "title": "list_overdue_invoicesArguments", "type": "object"}


With the real schema the model can see `region_code` and `min_amount_cents`, and a failed call
would now reach the model marked as an error. The same request goes through three more times.

In [20]:
for attempt in range(1, 4):
    AUDIT_LOG.clear()
    WIRE_RESULTS.clear()
    report_run(attempt, run_agent(connection, model_tools, read_tool_result, ANALYST_REQUEST))

  turn 1: list_overdue_invoices({'min_amount_cents': 500000, 'region_code': 'EMEA'}) -> 1 rows, failed=False


  turn 2: list_overdue_invoices({'min_amount_cents': 500000, 'region_code': 'EMEA'}) -> 1 rows, failed=False


attempt 1: 2 tool calls, client logged 0 failures, server sent isError on 0
answer: The total amount of overdue invoices in EMEA is 1,250,000 cents. The largest overdue invoice is INV-5001 from Nordlicht GmbH, due on 2026-07-31. This invoice is



  turn 1: list_overdue_invoices({'region_code': 'EMEA', 'min_amount_cents': 500000}) -> 1 rows, failed=False


  turn 2: list_overdue_invoices({'region_code': 'EMEA', 'min_amount_cents': 500000}) -> 1 rows, failed=False


attempt 2: 2 tool calls, client logged 0 failures, server sent isError on 0
answer: The total amount of overdue invoices in the EMEA region is 1,250,000 cents. The largest overdue invoice is INV-5001 from Nordlicht GmbH, amounting to 1,250,000 



  turn 1: list_overdue_invoices({'region_code': 'EMEA', 'min_amount_cents': 500000}) -> 1 rows, failed=False


  turn 2: list_overdue_invoices({'region_code': 'EMEA', 'min_amount_cents': 500000}) -> 1 rows, failed=False


  turn 3: list_overdue_invoices({'min_amount_cents': 500000, 'region_code': 'EMEA'}) -> 1 rows, failed=False


attempt 3: 3 tool calls, client logged 0 failures, server sent isError on 0
answer: The total number of overdue invoices in the EMEA region is 1. The total amount of overdue invoices is 1250000 cents. The due date for this invoice is 2026-07-31



Every call now sends `region_code` and `min_amount_cents` and gets back the one matching row, and
every answer names INV-5001 for 1250000 cents. The model sometimes repeats the same call, which
costs a turn but changes nothing, because the query only reads. Both rows below were printed by
the cells above.

| How the client reads the wire | Calls that failed | Failures the client logged | Answers that match the database |
|---|---|---|---|
| `input_schema` and `is_error` | 3 of 3 | 0 | 0 of 3 |
| `inputSchema` and `isError` | 0 of 7 | 0 | 3 of 3 |

## Step 7: Keep stdout for JSON-RPC and send every log line to stderr

Over stdio the server's stdout is the protocol channel, so any other text written there lands in
the middle of the stream of replies. Our server logs every query, and starting it with
`REPORTING_LOG=stdout` makes it log the way a plain `print` would.

![Keep stdout for JSON-RPC and send every log line to stderr](images/server-transport-step-3.svg)

In [21]:
noisy = StdioConnection(log_to="stdout")
start_session(noisy)
try:
    send_request(noisy, "tools/call",
                 {"name": "list_overdue_invoices", "arguments": {"region_code": "EMEA"}})
except ValueError as error:
    print(f"the client stopped: {error}")

reply_line = noisy.stdout_lines.get(timeout=10)
noisy.close()
print(f"the reply came next: {reply_line[:90]}...")

the client stopped: stdout carried a line that is not JSON-RPC: "[reporting] SELECT with ('EMEA', 0)"
the reply came next: {"jsonrpc":"2.0","id":2,"result":{"content":[{"type":"text","text":"{\n  \"invoice_id\": \...


The server's log line reached the client ahead of the reply, so the client stopped on it, and the
real reply arrived one line later with nothing left to read it. Over stdio, stdout belongs to
JSON-RPC alone, and a single log line written there breaks the connection.

The fix is **stderr isolation**: every log line goes to stderr, which the client never parses, so
stdout carries JSON-RPC and nothing else. That is the server's default, so the next cell starts it
without the override and reads stderr after it exits.

In [22]:
quiet = StdioConnection()
start_session(quiet)
stdio_rows = send_request(quiet, "tools/call", {"name": "list_overdue_invoices",
                                                "arguments": {"region_code": "EMEA"}})
log_text = quiet.close()

print(f"rows returned  : {len(stdio_rows['structuredContent']['result'])}")
print(f"stderr carried : {log_text.strip()}")

rows returned  : 2
stderr carried : [reporting] SELECT with ('EMEA', 0)


## Step 8: Reach the same server over the HTTP transport

A server that many clients share runs on its own machine, so no client can start it as a child
process. The **HTTP transport** sends each JSON-RPC message as an HTTP POST to a server that is
already running, and each request can carry an Authorization header, which stdio has nowhere to
put.

![Reach the same server over the HTTP transport](images/server-transport-step-4.svg)

In [23]:
def wait_for_port(port, seconds=10):
    """Poll until the server accepts connections, or give up at the deadline."""
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        try:
            socket.create_connection(("127.0.0.1", port), timeout=1).close()
            return
        except OSError:
            time.sleep(0.1)
    raise TimeoutError(f"nothing listening on port {port} after {seconds} seconds")

`HttpConnection` has the same `exchange` method as `StdioConnection`, so `send_request`,
`start_session` and `run_agent` work over either transport without a change.

In [24]:
class HttpConnection:
    """The same server run on its own, reached with one HTTP POST per JSON-RPC message."""

    def __init__(self, log_to="stderr"):
        with socket.socket() as probe:
            probe.bind(("127.0.0.1", 0))
            port = probe.getsockname()[1]
        env = {**os.environ, "REPORTING_LOG": log_to, "REPORTING_PORT": str(port),
               "PYTHONWARNINGS": "ignore"}
        self.process = subprocess.Popen(
            [sys.executable, "reporting_server.py", "streamable-http"], text=True, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        self.url, self.last_id = f"http://127.0.0.1:{port}/mcp", 0
        wait_for_port(port)

    def exchange(self, message):
        reply = requests.post(self.url, json=message, timeout=10,
                              headers={"Accept": "application/json, text/event-stream"})
        reply.raise_for_status()
        return reply.json() if "id" in message else None

    def close(self):
        self.process.terminate()
        return self.process.communicate(timeout=10)[0]

The next cell starts the HTTP server with its log on stdout, the same setting that broke the stdio
client in Step 7, and runs the same query.

In [25]:
remote = HttpConnection(log_to="stdout")
start_session(remote)
http_rows = send_request(remote, "tools/call", {"name": "list_overdue_invoices",
                                                "arguments": {"region_code": "EMEA"}})

print(f"same rows as over stdio : {http_rows['structuredContent'] == stdio_rows['structuredContent']}")

same rows as over stdio : True


The agent needs no change either, because it only ever calls `send_request`. The next cell runs it
once over HTTP, then stops the server and prints what it wrote to stdout.

In [26]:
AUDIT_LOG.clear()
WIRE_RESULTS.clear()
report_run("over HTTP", run_agent(remote, model_tools, read_tool_result, ANALYST_REQUEST))

print(f"server stdout:\n{remote.close().strip()}")

  turn 1: list_overdue_invoices({'region_code': 'EMEA', 'min_amount_cents': 500000}) -> 1 rows, failed=False


attempt over HTTP: 1 tool calls, client logged 0 failures, server sent isError on 0
answer: The total number of overdue invoices in the EMEA region is 1. The largest overdue invoice is for Nordlicht GmbH amounting to 12,500,000 cents, which is due on J

server stdout:
[reporting] SELECT with ('EMEA', 0)
[reporting] SELECT with ('EMEA', 500000)


Over HTTP the replies travel inside HTTP responses, so the server's stdout is an ordinary log
again, and both log lines above did no harm. Stderr isolation is a rule for stdio, where stdout is
the protocol channel. The answer itself gives the amount as 12,500,000 cents, ten times the 1250000
in the row the server returned, so the rows the client received remain the record to check an
answer against.

## Step 9: Test the integration without calling the model

Each safeguard above gets a test that runs in a few seconds with no API key, so it can run on
every commit. If someone reads `input_schema` again, prints a log line to stdout, or opens the
database for writing, one of these tests fails.

![Test the integration without calling the model](images/tool-calls-step-3.svg)

In [27]:
def test_translation_reads_input_schema():
    translated = translate_tool(MCP_TOOLS[0])
    assert translated["function"]["parameters"] == MCP_TOOLS[0]["inputSchema"]
    try:
        translate_tool({"name": "list_overdue_invoices", "input_schema": {}})
    except KeyError:
        return
    raise AssertionError("a tool with no inputSchema was offered to the model")


def test_is_error_marks_a_failed_call():
    failed_call = {"content": [{"type": "text", "text": "region is required"}], "isError": True}
    good_call = {"content": [{"type": "text", "text": "[]"}]}
    assert read_tool_result(failed_call)[1] is True, "isError was ignored"
    assert read_tool_result(good_call)[1] is False, "a call with no isError was flagged"

The last two tests start the real server, which is still quick because no model is involved.

In [28]:
def test_stdout_carries_only_json_rpc():
    server_process = StdioConnection()
    start_session(server_process)
    send_request(server_process, "tools/call",
                 {"name": "list_customer_invoices", "arguments": {"customer_id": "CUST-101"}})
    server_process.close()
    assert server_process.stdout_lines.empty(), "stdout carried a line after the last reply"


def test_server_refuses_writes():
    try:
        reporting_server.fetch_rows("DELETE FROM invoices", ())
    except sqlite3.OperationalError:
        assert count_rows("invoices") == 4
        return
    raise AssertionError("a write reached the reporting database")


for test in (test_translation_reads_input_schema, test_is_error_marks_a_failed_call,
             test_stdout_carries_only_json_rpc, test_server_refuses_writes):
    test()
    print(f"passed: {test.__name__}")

passed: test_translation_reads_input_schema
passed: test_is_error_marks_a_failed_call


[reporting] DELETE with ()


passed: test_stdout_carries_only_json_rpc
passed: test_server_refuses_writes


The last cell stops the stdio server and deletes every file this notebook wrote.

In [29]:
import shutil

connection.close()
for path in (DATABASE_PATH, pathlib.Path("reporting_server.py")):
    path.unlink(missing_ok=True)
shutil.rmtree("__pycache__", ignore_errors=True)
print("stopped the server and removed reporting.db, reporting_server.py and __pycache__")

stopped the server and removed reporting.db, reporting_server.py and __pycache__


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **MCP primitives** | `@server.tool`, `@server.resource`, `@server.prompt` | Tools the model calls, resources the client reads, prompts a person picks |
| **JSON-RPC 2.0** | `send_request` and `start_session` | One request per line, matched to its reply by `id` |
| **stdio transport** | `StdioConnection` | Starts the server as a child process and talks over its stdin and stdout |
| **HTTP transport** | `HttpConnection` | Posts the same messages to a server that is already running |
| **Wire field names** | `translate_tool` and `read_tool_result` | Read `inputSchema` and `isError` exactly as the protocol spells them |
| **stderr isolation** | `log_query` in `reporting_server.py` | Keeps log lines off stdout, so the reply stream stays valid |
| **Read only access** | `mode=ro` in `fetch_rows` | The driver refuses any write, whatever a tool asks for |